# (27) Jobs: beta fits

**Motivation**: Make $\beta$ job runnsers as txt file. <br>

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-vae/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-vae/figs')
tmp_dir = os.path.join(git_dir, 'jb-vae/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, 'PoissonVAE'))
from analysis.eval import sparse_score
from figures.fighelper import *
from vae.train_vae import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

ModuleNotFoundError: No module named 'vae'

## Setup

In [2]:
from analysis.helper import job_runner_script


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"


def divide_list(lst, n):
    k, m = divmod(len(lst), n)
    return [lst[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n)]

In [3]:
save_dir = 'Dropbox/git/PoissonVAE/scripts'
save_dir = pjoin(os.environ['HOME'], save_dir)
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

['fit_vae.sh', 'kill_screens.sh', 'resume_fit.sh', 'run_sessions.sh']


## Betas (mach)

```<lin|lin>```

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts_mach = collections.defaultdict(list)
tot = 0

In [5]:
n_seeds = 5
seeds = range(1, n_seeds + 1)

betas = [
    0.001, 0.01, 0.2, 0.4, 0.6, 0.8,
    1.0,
    1.2, 2.0, 3.0, 4.0, 5.0,
]

In [6]:
combos = itertools.product(enumerate(betas), seeds)
for (idx, b), s in combos:
    arg = ' '.join([
        f"--kl_beta {b}",
        f"--comment b{b:0.2g}",
    ])
    gpu_i = idx // 3

    kws = dict(
        device=gpu_i,
        dataset='vH16',
        model='poisson',
        archi='lin|lin',
        seed=s,
        args=arg,
    )
    scripts_mach[gpu_i].append(job_runner_script(**kws))
    tot += 1

In [7]:
print(tot)

60

In [8]:
scripts_mach = dict(scripts_mach)
print({k: len(v) for k, v in scripts_mach.items()})

{0: 15, 1: 15, 2: 15, 3: 15}

### Save

In [9]:
n_fits = 5

for gpu_i, scripts in scripts_mach.items():
    scripts_divided = divide_list(scripts, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )

[PROGRESS] 'mach-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda0-fit3.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda0-fit4.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda1-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda1-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda1-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda1-fit3.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda1-fit4.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda2-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda2-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 5.0 --comment b5 && 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 5.0 --comment b5 && 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 5.0 --comment b5

In [11]:
print(scripts_mach)

{
    0: [
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 0.001 --comment b0.001",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 0.001 --comment b0.001",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 0.001 --comment b0.001",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 0.001 --comment b0.001",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 0.001 --comment b0.001",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 0.01 --comment b0.01",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 0.01 --comment b0.01",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 0.01 --comment b0.01",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 0.01 --comment b0.01",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 0.01 --comment b0.01",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 0.2 --comment b0.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 0.2 --comment b0.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 0.2 --comment b0.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 0.2 --comment b0.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 0.2 --comment b0.2"
    ],
    1: [
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 0.4 --comment b0.4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 0.4 --comment b0.4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 0.4 --comment b0.4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 0.4 --comment b0.4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 0.4 --comment b0.4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 0.6 --comment b0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 0.6 --comment b0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 0.6 --comment b0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 0.6 --comment b0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 0.6 --comment b0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 0.8 --comment b0.8",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 0.8 --comment b0.8",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 0.8 --comment b0.8",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 0.8 --comment b0.8",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 0.8 --comment b0.8"
    ],
    2: [
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 1.0 --comment b1",
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 1.0 --comment b1",
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 1.0 --comment b1",
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 1.0 --comment b1",
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 1.0 --comment b1",
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 1.2 --comment b1.2",
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 1.2 --comment b1.2",
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 1.2 --comment b1.2",
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 1.2 --comment b1.2",
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 1.2 --comment b1.2",
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 2.0 --comment b2",
        "./fit_vae.sh '2' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 2.0 --comment b2",
        "./fit_vae.sh '2' 'vH16' 'poi

In [12]:
print(scripts_divided)

[
    [
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 3.0 --comment b3",
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 3.0 --comment b3",
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 3.0 --comment b3"
    ],
    [
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 3.0 --comment b3",
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 3.0 --comment b3",
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 4.0 --comment b4"
    ],
    [
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 4.0 --comment b4",
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 4.0 --comment b4",
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 4.0 --comment b4"
    ],
    [
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 4.0 --comment b4",
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 1 --kl_beta 5.0 --comment b5",
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 2 --kl_beta 5.0 --comment b5"
    ],
    [
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 3 --kl_beta 5.0 --comment b5",
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 4 --kl_beta 5.0 --comment b5",
        "./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 5 --kl_beta 5.0 --comment b5"
    ]
]

## Betas (chewie + yoru)

```<conv|lin>```

### chewie

In [4]:
host = 'chewie'
_cleanup(save_dir, host)

scripts_chewie = collections.defaultdict(list)
tot = 0

In [5]:
n_seeds = 5
seeds = range(1, n_seeds + 1)

betas = [0.001, 0.01, 0.2, 0.4, 0.6, 0.8]

In [6]:
combos = itertools.product(enumerate(betas), seeds)
for (idx, b), s in combos:
    arg = ' '.join([
        f"--kl_beta {b}",
        f"--comment b{b:0.2g}",
    ])
    gpu_i = idx // 3

    kws = dict(
        device=gpu_i,
        dataset='vH16',
        model='poisson',
        archi='conv+b|lin',
        seed=s,
        args=arg,
    )
    scripts_chewie[gpu_i].append(job_runner_script(**kws))
    tot += 1

In [7]:
print(tot)

30

In [8]:
scripts_chewie = dict(scripts_chewie)
print({k: len(v) for k, v in scripts_chewie.items()})

{0: 15, 1: 15}

#### Save

In [9]:
n_fits = 5

for gpu_i, scripts in scripts_chewie.items():
    scripts_divided = divide_list(scripts, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )

[PROGRESS] 'chewie-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'chewie-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'chewie-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'chewie-cuda0-fit3.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'chewie-cuda0-fit4.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'chewie-cuda1-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'chewie-cuda1-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'chewie-cuda1-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'chewie-cuda1-fit3.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'chewie-cuda1-fit4.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 0.8 --comment b0.8 && 
./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 0.8 --comment b0.8 && 
./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 0.8 --comment b0.8

In [11]:
print(scripts_chewie)

{
    0: [
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.001 --comment b0.001",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 0.001 --comment b0.001",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 0.001 --comment b0.001",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 0.001 --comment b0.001",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 0.001 --comment b0.001",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.01 --comment b0.01",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 0.01 --comment b0.01",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 0.01 --comment b0.01",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 0.01 --comment b0.01",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 0.01 --comment b0.01",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.2 --comment b0.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 0.2 --comment b0.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 0.2 --comment b0.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 0.2 --comment b0.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 0.2 --comment b0.2"
    ],
    1: [
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.4 --comment b0.4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 0.4 --comment b0.4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 0.4 --comment b0.4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 0.4 --comment b0.4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 0.4 --comment b0.4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.6 --comment b0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 0.6 --comment b0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 0.6 --comment b0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 0.6 --comment b0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 0.6 --comment b0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.8 --comment b0.8",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 0.8 --comment b0.8",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 0.8 --comment b0.8",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 0.8 --comment b0.8",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 0.8 --comment b0.8"
    ]
}

### yoru

In [12]:
host = 'yoru'
_cleanup(save_dir, host)

scripts_yoru = collections.defaultdict(list)
tot = 0

In [13]:
n_seeds = 5
seeds = range(1, n_seeds + 1)

betas = [1.0, 1.2, 2.0, 3.0, 4.0, 5.0]

In [14]:
combos = itertools.product(enumerate(betas), seeds)
for (idx, b), s in combos:
    arg = ' '.join([
        f"--kl_beta {b}",
        f"--comment b{b:0.2g}",
    ])
    gpu_i = idx // 3

    kws = dict(
        device=gpu_i,
        dataset='vH16',
        model='poisson',
        archi='conv+b|lin',
        seed=s,
        args=arg,
    )
    scripts_yoru[gpu_i].append(job_runner_script(**kws))
    tot += 1

In [15]:
print(tot)

30

In [16]:
scripts_yoru = dict(scripts_yoru)
print({k: len(v) for k, v in scripts_yoru.items()})

{0: 15, 1: 15}

#### Save

In [17]:
n_fits = 5

for gpu_i, scripts in scripts_yoru.items():
    scripts_divided = divide_list(scripts, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )

[PROGRESS] 'yoru-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'yoru-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'yoru-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'yoru-cuda0-fit3.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'yoru-cuda0-fit4.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'yoru-cuda1-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'yoru-cuda1-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'yoru-cuda1-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'yoru-cuda1-fit3.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'yoru-cuda1-fit4.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


Print one to check

In [18]:
print(combined.replace('&& ', '&& \n'))

./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 5.0 --comment b5 && 
./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 5.0 --comment b5 && 
./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 5.0 --comment b5

In [19]:
print(scripts_yoru)

{
    0: [
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 1.0 --comment b1",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 1.0 --comment b1",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 1.0 --comment b1",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 1.0 --comment b1",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 1.0 --comment b1",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 1.2 --comment b1.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 1.2 --comment b1.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 1.2 --comment b1.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 1.2 --comment b1.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 1.2 --comment b1.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 2.0 --comment b2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 2.0 --comment b2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 2.0 --comment b2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 2.0 --comment b2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 2.0 --comment b2"
    ],
    1: [
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 3.0 --comment b3",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 3.0 --comment b3",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 3.0 --comment b3",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 3.0 --comment b3",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 3.0 --comment b3",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 4.0 --comment b4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 4.0 --comment b4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 4.0 --comment b4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 4.0 --comment b4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 4.0 --comment b4",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 5.0 --comment b5",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 2 --kl_beta 5.0 --comment b5",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 3 --kl_beta 5.0 --comment b5",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 4 --kl_beta 5.0 --comment b5",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 5 --kl_beta 5.0 --comment b5"
    ]
}